# Priced Level 3 Cluster Analysis (AWS Embeddings)

This notebook mirrors the original clustering analysis, but uses AWS Titan embeddings created from the exact text concatenation logic used in classifier training (`build_product_text`).

Original notebook remains unchanged:
- `analysis/notebooks/Priced Level 3 Cluster Analysis.ipynb`

This version supports:
- cache-aware embedding generation
- PCA component experimentation
- KMeans silhouette comparison across PCA settings

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics import silhouette_score

# Resolve project root so notebook imports work even when launched from analysis/notebooks.
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "product_classifier_utils.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "product_classifier_utils.py").exists():
    raise FileNotFoundError("Could not locate project root containing product_classifier_utils.py")

import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from product_classifier_utils import (  # noqa: E402
    build_product_text,
    embed_texts_from_cache,
    get_bedrock_client,
    get_snowflake_session,
    load_product_data,
    stable_text_hash,
)

: 

In [ ]:
# Core run configuration
TABLE = "SNOWFLAKE_LEARNING_DB.SMCMAHON_PRODUCTS.PRODUCTS_L3_SUB"
LABEL_COLUMN = "PARENT_3_CATEGORY"
MIN_CATEGORY_COUNT = 10
ONLY_PRICED = True
ROW_LIMIT = None

# Embedding + cache settings
AWS_PROFILE = "staging.admin"
AWS_REGION = "us-east-1"
MODEL_ID = "amazon.titan-embed-text-v1"
MAX_WORKERS = 16
CHECKPOINT_EVERY = 5000
MAX_RETRIES = 5
CACHE_PATH = PROJECT_ROOT / "artifacts/cache/embedding_cache.pkl"

# LLM labeling settings
LLM_MODEL_ID = "anthropic.claude-3-5-sonnet-20240620-v1:0"
LLM_MAX_CATEGORIES = 10
LLM_TEMPERATURE = 0.2

# Clustering experiment settings
PCA_COMPONENTS_TO_TRY = [None, 256, 512, 768]
K_VALUES_TO_TRY = [15, 16, 17, 18, 19, 20]
SAMPLE_SIZE_FOR_SILHOUETTE = 30000
RANDOM_STATE = 42

# Output settings
SAVE_LOCAL_CSV = True
LOCAL_OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "analysis"
SAVE_TO_SNOWFLAKE_TABLE = False
OUTPUT_TABLE_NAME = "PROPOSED_L3_AWS"

# Reproducible cluster bundle settings (for scoring unseen data later)
SAVE_CLUSTER_BUNDLE = True
CLUSTER_BUNDLE_NAME = "l3_aws_cluster_bundle"

In [ ]:
# Load source data from Snowflake using the same min-category logic used in training.
sf_session = get_snowflake_session()
df = load_product_data(
    session=sf_session,
    table=TABLE,
    label_column=LABEL_COLUMN,
    min_category_count=MIN_CATEGORY_COUNT,
    row_limit=ROW_LIMIT,
)

if ONLY_PRICED:
    df = df[df["PRICING_STATUS_C"].astype(str).str.upper() == "PRICED"].reset_index(drop=True)

print(f"Loaded rows: {len(df):,}")
print("Columns:", sorted(df.columns)[:12], "...")
print(df[[LABEL_COLUMN, "PRICING_STATUS_C"]].head(3))

In [ ]:
# Build classifier-consistent text and embed via AWS Titan with cache reuse.
def load_cache(path: Path) -> dict:
    if not path.exists():
        return {}
    with path.open("rb") as f:
        obj = pickle.load(f)
    if not isinstance(obj, dict):
        raise ValueError(f"Cache file {path} is not a dict")
    return obj


def save_cache(path: Path, cache: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("wb") as f:
        pickle.dump(cache, f)


text_series = build_product_text(df)
texts = text_series.tolist()
text_hashes = [stable_text_hash(t) for t in texts]

cache = load_cache(CACHE_PATH)
cache_before = len(cache)
client = get_bedrock_client(profile_name=AWS_PROFILE, region=AWS_REGION)


def checkpoint_cb(cache_obj: dict, processed_count: int) -> None:
    save_cache(CACHE_PATH, cache_obj)
    print(f"Checkpointed {processed_count:,} new embeddings. Cache size={len(cache_obj):,}")


X = embed_texts_from_cache(
    texts=texts,
    text_hashes=text_hashes,
    cache=cache,
    client=client,
    model_id=MODEL_ID,
    show_progress=True,
    max_workers=MAX_WORKERS,
    checkpoint_every=CHECKPOINT_EVERY if CHECKPOINT_EVERY > 0 else None,
    on_checkpoint=checkpoint_cb if CHECKPOINT_EVERY > 0 else None,
    max_retries=MAX_RETRIES,
)

save_cache(CACHE_PATH, cache)
print(f"Embedding matrix shape: {X.shape}")
print(f"Cache size: {cache_before:,} -> {len(cache):,}")

df["TEXT_TO_EMBED"] = text_series

In [ ]:
# Evaluate KMeans silhouette across multiple PCA dimensions.
rng = np.random.default_rng(RANDOM_STATE)
sample_n = min(SAMPLE_SIZE_FOR_SILHOUETTE, len(X))
sample_idx = rng.choice(len(X), size=sample_n, replace=False)

rows = []
transformed_by_pca = {}
pca_model_by_label = {}

for pca_components in PCA_COMPONENTS_TO_TRY:
    if pca_components is None:
        X_eval = X
        pca_label = "none"
        explained = np.nan
        pca_model = None
    else:
        pca_model = PCA(n_components=pca_components, random_state=RANDOM_STATE)
        X_eval = pca_model.fit_transform(X).astype(np.float32)
        pca_label = str(pca_components)
        explained = float(np.sum(pca_model.explained_variance_ratio_))

    transformed_by_pca[pca_label] = X_eval
    pca_model_by_label[pca_label] = pca_model
    X_sample = X_eval[sample_idx]

    for k in K_VALUES_TO_TRY:
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init="auto", max_iter=300)
        labels = km.fit_predict(X_sample)
        sil = silhouette_score(X_sample, labels)
        rows.append(
            {
                "pca": pca_label,
                "pca_components": pca_components,
                "k": k,
                "silhouette": float(sil),
                "explained_variance_sum": explained,
            }
        )

results_df = pd.DataFrame(rows).sort_values("silhouette", ascending=False).reset_index(drop=True)
results_df.head(20)

In [ ]:
# Pick a final PCA/k config from results and cluster full data.
# Example: choose best row automatically; or set FINAL_PCA / FINAL_K manually.
BEST = results_df.iloc[0]
FINAL_PCA = BEST["pca"]  # "none" or stringified integer
FINAL_K = int(BEST["k"])

print("Chosen config:", {"pca": FINAL_PCA, "k": FINAL_K, "silhouette": float(BEST["silhouette"])})

X_final = transformed_by_pca[str(FINAL_PCA)]
selected_pca_model = pca_model_by_label[str(FINAL_PCA)]
km_final = KMeans(n_clusters=FINAL_K, random_state=RANDOM_STATE, n_init="auto", max_iter=300)
df["CLUSTER"] = km_final.fit_predict(X_final).astype("int32")

df["CLUSTER"].value_counts().sort_index()

In [ ]:
# Extract top TF-IDF terms per cluster for interpretation.
DOMAIN_STOPWORDS = {
    "price",
    "pricing",
    "status",
    "quote",
    "quoted",
    "description",
    "insert",
    "item",
    "items",
    "product",
    "products",
    "unit",
    "units",
}
STOPWORDS = list(ENGLISH_STOP_WORDS.union(DOMAIN_STOPWORDS))

vectorizer = TfidfVectorizer(
    max_features=20000,
    stop_words=STOPWORDS,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.8,
)

tfidf_matrix = vectorizer.fit_transform(df["TEXT_TO_EMBED"])
feature_names = np.array(vectorizer.get_feature_names_out())


def top_terms_per_cluster(tfidf, dataf, features, top_n=15):
    out = {}
    for c in sorted(dataf["CLUSTER"].unique()):
        mask = (dataf["CLUSTER"] == c).values
        centroid = np.asarray(tfidf[mask].mean(axis=0)).ravel()
        top_idx = centroid.argsort()[::-1][:top_n]
        out[int(c)] = features[top_idx].tolist()
    return out


cluster_keywords = top_terms_per_cluster(tfidf_matrix, df, feature_names, top_n=15)
cluster_keywords

In [ ]:
# Optional: persist results for downstream review / taxonomy workshops.
run_tag = f"l3_priced_aws_pca_{FINAL_PCA}_k_{FINAL_K}"
out_dir = LOCAL_OUTPUT_DIR / run_tag
out_dir.mkdir(parents=True, exist_ok=True)

results_df.to_csv(out_dir / "pca_k_silhouette_comparison.csv", index=False)
(df[["PRODUCT_ID", LABEL_COLUMN, "PRICING_STATUS_C", "TEXT_TO_EMBED", "CLUSTER"]]
   .to_csv(out_dir / "cluster_assignments.csv", index=False))

pd.DataFrame(
    [{"cluster": c, "top_terms": ", ".join(terms)} for c, terms in cluster_keywords.items()]
).to_csv(out_dir / "cluster_top_terms.csv", index=False)

if SAVE_LOCAL_CSV:
    sme_share_path = out_dir / "cluster_assignments_for_sme_review.csv"
    cols = [
        "PRODUCT_ID",
        LABEL_COLUMN,
        "PRICING_STATUS_C",
        "TEXT_TO_EMBED",
        "CLUSTER",
    ]
    if "PROPOSED_CATEGORY" in df.columns:
        cols.append("PROPOSED_CATEGORY")
    df[cols].to_csv(sme_share_path, index=False)
    print(f"Saved SME share CSV: {sme_share_path}")

if SAVE_TO_SNOWFLAKE_TABLE:
    snow_df = sf_session.create_dataframe(df)
    snow_df.write.mode("overwrite").save_as_table(OUTPUT_TABLE_NAME)
    print(f"Wrote Snowflake table: {OUTPUT_TABLE_NAME}")

print(f"Saved analysis artifacts to: {out_dir}")

## Optional: LLM-assisted category label suggestions

This section proposes cluster-to-category labels from `cluster_keywords` using a Bedrock chat model. The output is JSON so you can review/edit before applying.

Recommended workflow:
1. Run suggestion cell
2. Inspect/edit mapping in the next cell
3. Apply mapping to dataframe
4. Export assignments for taxonomy review

In [ ]:
import json


def suggest_cluster_labels_with_bedrock(
    cluster_terms: dict,
    llm_model_id: str = LLM_MODEL_ID,
    max_categories: int = LLM_MAX_CATEGORIES,
    temperature: float = LLM_TEMPERATURE,
):
    """Return JSON suggestions for cluster -> category labels."""
    prompt = f"""
You are helping design a product taxonomy.
Given cluster keyword lists, propose concise category labels.

Rules:
- Return strict JSON only.
- Use clear business-facing labels.
- Keep labels mutually distinct and reusable.
- Prefer consolidating clusters into <= {max_categories} categories when reasonable.
- Do not use markdown.

Input:
{json.dumps(cluster_terms, indent=2)}

Return JSON object with keys:
- cluster_to_label: object mapping cluster id (as string) -> label
- label_descriptions: object mapping label -> one-sentence description
"""

    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 1600,
        "temperature": temperature,
        "messages": [{"role": "user", "content": prompt}],
    }

    llm_client = get_bedrock_client(profile_name=AWS_PROFILE, region=AWS_REGION)
    response = llm_client.invoke_model(
        modelId=llm_model_id,
        body=json.dumps(body),
        contentType="application/json",
        accept="application/json",
    )
    payload = json.loads(response["body"].read().decode("utf-8"))
    text = payload["content"][0]["text"]
    return json.loads(text)


label_suggestions = suggest_cluster_labels_with_bedrock(
    cluster_keywords,
    llm_model_id=LLM_MODEL_ID,
    max_categories=LLM_MAX_CATEGORIES,
    temperature=LLM_TEMPERATURE,
)
label_suggestions

In [ ]:
# Review/edit this mapping before applying.
# Start from LLM proposal and then customize as needed.
cluster_to_label = {
    int(k): v for k, v in label_suggestions["cluster_to_label"].items()
}

# Example manual override:
# cluster_to_label[3] = "Nucleic Acid Reagents"

missing = sorted(set(df["CLUSTER"].unique()) - set(cluster_to_label.keys()))
if missing:
    raise ValueError(f"Missing cluster labels for clusters: {missing}")

df["PROPOSED_CATEGORY"] = df["CLUSTER"].map(cluster_to_label)
assert df["PROPOSED_CATEGORY"].isna().sum() == 0

pd.Series(cluster_to_label).sort_index()

In [ ]:
# Optional: save suggested labels + applied mapping for governance / handoff.
label_out = out_dir / "llm_label_suggestions.json"
with label_out.open("w", encoding="utf-8") as f:
    json.dump(label_suggestions, f, indent=2)

mapping_out = out_dir / "cluster_to_category_mapping.csv"
pd.DataFrame(
    [{"cluster": int(k), "proposed_category": v} for k, v in sorted(cluster_to_label.items())]
).to_csv(mapping_out, index=False)

final_assignments_out = out_dir / "cluster_assignments_with_categories.csv"
(df[["PRODUCT_ID", LABEL_COLUMN, "CLUSTER", "PROPOSED_CATEGORY", "TEXT_TO_EMBED"]]
   .to_csv(final_assignments_out, index=False))

print("Saved:")
print("-", label_out)
print("-", mapping_out)
print("-", final_assignments_out)

In [ ]:
# Save reproducible clustering bundle for future unseen-data scoring.
if SAVE_CLUSTER_BUNDLE:
    bundle_dir = out_dir / CLUSTER_BUNDLE_NAME
    bundle_dir.mkdir(parents=True, exist_ok=True)

    with (bundle_dir / "kmeans.pkl").open("wb") as f:
        pickle.dump(km_final, f)

    with (bundle_dir / "pca.pkl").open("wb") as f:
        pickle.dump(selected_pca_model, f)

    # If label mapping exists, save it. Otherwise save cluster IDs only.
    if "cluster_to_label" in globals():
        mapping_to_save = {int(k): str(v) for k, v in cluster_to_label.items()}
    else:
        mapping_to_save = {int(c): f"cluster_{int(c)}" for c in sorted(df["CLUSTER"].unique())}

    with (bundle_dir / "cluster_to_label_mapping.json").open("w", encoding="utf-8") as f:
        json.dump(mapping_to_save, f, indent=2)

    run_config = {
        "table": TABLE,
        "label_column": LABEL_COLUMN,
        "min_category_count": MIN_CATEGORY_COUNT,
        "only_priced": ONLY_PRICED,
        "row_limit": ROW_LIMIT,
        "embedding_model_id": MODEL_ID,
        "aws_region": AWS_REGION,
        "final_pca": str(FINAL_PCA),
        "final_k": int(FINAL_K),
        "random_state": int(RANDOM_STATE),
        "run_tag": run_tag,
    }
    with (bundle_dir / "run_config.json").open("w", encoding="utf-8") as f:
        json.dump(run_config, f, indent=2)

    print(f"Saved cluster bundle to: {bundle_dir}")
    print("-", bundle_dir / "kmeans.pkl")
    print("-", bundle_dir / "pca.pkl")
    print("-", bundle_dir / "cluster_to_label_mapping.json")
    print("-", bundle_dir / "run_config.json")

In [ ]:
# Helper: load saved bundle and score unseen dataframe.
def score_with_cluster_bundle(new_df: pd.DataFrame, bundle_dir: Path) -> pd.DataFrame:
    bundle_dir = Path(bundle_dir)

    with (bundle_dir / "kmeans.pkl").open("rb") as f:
        kmeans_model = pickle.load(f)
    with (bundle_dir / "pca.pkl").open("rb") as f:
        pca_model = pickle.load(f)
    with (bundle_dir / "cluster_to_label_mapping.json").open("r", encoding="utf-8") as f:
        c2l = {int(k): v for k, v in json.load(f).items()}

    new_text = build_product_text(new_df)
    new_hashes = [stable_text_hash(t) for t in new_text.tolist()]

    cache_local = load_cache(CACHE_PATH)
    client_local = get_bedrock_client(profile_name=AWS_PROFILE, region=AWS_REGION)
    X_new = embed_texts_from_cache(
        texts=new_text.tolist(),
        text_hashes=new_hashes,
        cache=cache_local,
        client=client_local,
        model_id=MODEL_ID,
        show_progress=True,
        max_workers=MAX_WORKERS,
        checkpoint_every=CHECKPOINT_EVERY if CHECKPOINT_EVERY > 0 else None,
        on_checkpoint=checkpoint_cb if CHECKPOINT_EVERY > 0 else None,
        max_retries=MAX_RETRIES,
    )
    save_cache(CACHE_PATH, cache_local)

    if pca_model is not None:
        X_new = pca_model.transform(X_new).astype(np.float32)

    cluster_ids = kmeans_model.predict(X_new).astype(int)
    scored = new_df.copy()
    scored["NEW_CLUSTER"] = cluster_ids
    scored["NEW_CATEGORY"] = [c2l.get(int(c), f"cluster_{int(c)}") for c in cluster_ids]
    return scored

# Example usage:
# unseen_df = pd.read_csv("path/to/new_products.csv")
# bundle_path = out_dir / CLUSTER_BUNDLE_NAME
# unseen_scored = score_with_cluster_bundle(unseen_df, bundle_path)
# unseen_scored.head()